# 01 — Setup & data

Clone **musicinstruct** + **midi-agent**, restore the MIDI-Instruct dataset from Drive, validate, and run a quick **oracle** smoke test.

**CPU OK.**

In [ ]:
# @title Configuration
MUSICINSTRUCT_REPO = "https://github.com/juliagsy/musicinstruct.git"
MIDI_AGENT_REPO = "https://github.com/juliagsy/midi-agent.git"
BRANCH = "main"  # @param {type:"string"}

WORK = "/content"
MUSICINSTRUCT = f"{WORK}/musicinstruct"
MIDI_AGENT = f"{WORK}/midi-agent"

DATASET_MODE = "real_v0.2"  # @param ["pilot", "real_v0.2"]
RUN_ID = "real_v0.2_v1" if DATASET_MODE == "real_v0.2" else "pilot_v1"  # @param {type:"string"}
SPLIT = "test"  # @param ["test", "validation"]
SMOKE_ITEMS = 5  # @param {type:"integer"}  0 to skip oracle smoke

LOCAL_DATA = f"{MUSICINSTRUCT}/data/pilot" if DATASET_MODE == "pilot" else f"{MUSICINSTRUCT}/data/v0.2"
MANIFEST_NAME = "pilot.jsonl" if DATASET_MODE == "pilot" else "manifest.jsonl"
MANIFEST = f"{LOCAL_DATA}/{MANIFEST_NAME}"

USE_DRIVE = True  # @param {type:"boolean"}
DRIVE_MI = "/content/drive/MyDrive/musicinstruct"
DRIVE_DATA = f"{DRIVE_MI}/data/pilot" if DATASET_MODE == "pilot" else f"{DRIVE_MI}/data/v0.2"

In [ ]:
# @title Bootstrap & restore dataset
import json
import shutil
import subprocess
from pathlib import Path

def sh(cmd: str) -> None:
    subprocess.run(cmd, shell=True, check=True)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

for path, url in [(MUSICINSTRUCT, MUSICINSTRUCT_REPO), (MIDI_AGENT, MIDI_AGENT_REPO)]:
    if Path(path).is_dir():
        shutil.rmtree(path)
    sh(f"git clone --branch {BRANCH} --depth 1 {url} {path}")

sh(f"pip install -q -e {MUSICINSTRUCT}")
sh(f"pip install -q -e '{MIDI_AGENT}[llm,dev]'")

if USE_DRIVE:
    if not (Path(DRIVE_DATA) / MANIFEST_NAME).is_file():
        raise FileNotFoundError(
            f"Missing {DRIVE_DATA}. Run musicinstruct colab/01_setup_and_data first."
        )
    Path(LOCAL_DATA).mkdir(parents=True, exist_ok=True)
    shutil.copytree(DRIVE_DATA, LOCAL_DATA, dirs_exist_ok=True)
    print("Restored from", DRIVE_DATA)

sh(f"musicinstruct validate {MANIFEST}")
print("manifest:", MANIFEST)

In [ ]:
# @title Oracle smoke
import json
import subprocess
from pathlib import Path

OUT = f"{MIDI_AGENT}/results/smoke_oracle"
if Path(OUT).exists():
    shutil.rmtree(OUT)

cmd = ["midi-agent", "run", MANIFEST, "--variant", "oracle", "--output-dir", OUT, "--split", SPLIT]
if SMOKE_ITEMS:
    cmd.extend(["--max-items", str(SMOKE_ITEMS)])
subprocess.run(cmd, check=True, cwd=MIDI_AGENT)
print(json.dumps(json.loads(Path(OUT).joinpath("agent_report.json").read_text()), indent=2))